In [ ]:
import pandas as pd
import numpy as np
import json
import sqlite3

from pathlib import Path
import sys

def find_project_root(start: Path, marker: str = "src") -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / marker).exists():
            return path
    raise RuntimeError(f"Could not find project root containing '{marker}'")

PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
base = PROJECT_ROOT / "experiments/completed/exp_004_project04_final/strategy_timeseries"

import pandas as pd

df_ls30 = pd.read_csv(base / "ls_30pct.csv")
df_blend = pd.read_csv(base / "blend_60_40_ls20_plus_ls30.csv")

returns_ls30 = df_ls30["portfolio_return"]
returns_blend = df_blend["portfolio_return"]

equity_ls30 = df_ls30["equity_curve"]
equity_blend = df_blend["equity_curve"]

In [ ]:
# Blend 60/40
from src.risk.drawdown import compute_drawdown, drawdown_exposure, apply_exposure_to_return

dd_blend = compute_drawdown((1 + returns_blend).cumprod())
exp_blend = drawdown_exposure(dd_blend)
ret_blend_dd = apply_exposure_to_return(returns_blend, exp_blend)

# LS 30%
dd_ls30 = compute_drawdown((1 + returns_ls30).cumprod())
exp_ls30 = drawdown_exposure(dd_ls30)
ret_ls30_dd = apply_exposure_to_return(returns_ls30, exp_ls30)

In [ ]:
equity_blend_dd = (1 + ret_blend_dd).cumprod()
equity_ls30_dd = (1 + ret_ls30_dd).cumprod()

In [ ]:
from src.utils.metrics import sharpe_ratio, max_drawdown

def compute_cagr(equity_curve):
    total_return = equity_curve.iloc[-1] / equity_curve.iloc[0]
    n_years = len(equity_curve) / 252
    return total_return ** (1 / n_years) - 1

def annual_vol(returns):
    return returns.std() * np.sqrt(252)

def calmar_ratio(equity_curve):
    cagr = compute_cagr(equity_curve)
    mdd = abs(max_drawdown(equity_curve))
    return cagr / mdd if mdd != 0 else np.nan

In [ ]:
import numpy as np

comparison = pd.DataFrame([
    {
        "Strategy": "Blend 60/40 Baseline",
        "Sharpe": sharpe_ratio(returns_blend),
        "MDD": max_drawdown(equity_blend),
        "CAGR": compute_cagr(equity_blend),
        "Vol": annual_vol(returns_blend),
        "Calmar": calmar_ratio(equity_blend),
    },
    {
        "Strategy": "Blend 60/40 + DD Control",
        "Sharpe": sharpe_ratio(ret_blend_dd),
        "MDD": max_drawdown(equity_blend_dd),
        "CAGR": compute_cagr(equity_blend_dd),
        "Vol": annual_vol(ret_blend_dd),
        "Calmar": calmar_ratio(equity_blend_dd),
    },
    {
        "Strategy": "LS 30% Baseline",
        "Sharpe": sharpe_ratio(returns_ls30),
        "MDD": max_drawdown(df_ls30["equity_curve"]),
        "CAGR": compute_cagr(df_ls30["equity_curve"]),
        "Vol": annual_vol(returns_ls30),
        "Calmar": calmar_ratio(df_ls30["equity_curve"]),
    },
    {
        "Strategy": "LS 30% + DD Control",
        "Sharpe": sharpe_ratio(ret_ls30_dd),
        "MDD": max_drawdown(equity_ls30_dd),
        "CAGR": compute_cagr(equity_ls30_dd),
        "Vol": annual_vol(ret_ls30_dd),
        "Calmar": calmar_ratio(equity_ls30_dd),
    },
]).round(3)

comparison

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(df_blend["equity_curve"], label="Blend 60/40 Baseline")
plt.plot(equity_blend_dd, label="Blend 60/40 + DD Control")
plt.plot(df_ls30["equity_curve"], label="LS 30% Baseline")
plt.plot(equity_ls30_dd, label="LS 30% + DD Control")
plt.legend()
plt.title("Project 05: Drawdown Overlay Comparison")
plt.xlabel("Time")
plt.ylabel("Equity Curve")
plt.show()

**Testing - Other Strategies**

In [ ]:
strategy_base = PROJECT_ROOT / "experiments/completed/exp_004_project04_final/strategy_timeseries"

strategy_files = sorted(strategy_base.glob("*.csv"))

all_strategies = {}

for file in strategy_files:
    df = pd.read_csv(file)
    all_strategies[file.stem] = df

print("Loaded strategies:")
for k in all_strategies.keys():
    print("-", k)

In [ ]:
all_results = []

for name, df in all_strategies.items():
    returns = df["portfolio_return"].fillna(0)
    equity = (1 + returns).cumprod()

    dd = compute_drawdown(equity)
    exp = drawdown_exposure(dd)
    ret_dd = apply_exposure_to_return(returns, exp)
    equity_dd = (1 + ret_dd).cumprod()

    all_results.append({
        "Strategy": name,
        "Sharpe_base": sharpe_ratio(returns),
        "MDD_base": max_drawdown(equity),
        "CAGR_base": compute_cagr(equity),
        "Vol_base": annual_vol(returns),
        "Calmar_base": calmar_ratio(equity),

        "Sharpe_dd": sharpe_ratio(ret_dd),
        "MDD_dd": max_drawdown(equity_dd),
        "CAGR_dd": compute_cagr(equity_dd),
        "Vol_dd": annual_vol(ret_dd),
        "Calmar_dd": calmar_ratio(equity_dd),

        "Avg_Exposure": exp.shift(1).fillna(1.0).mean(),
    })

In [ ]:
all_results_df = pd.DataFrame(all_results).round(3)

all_results_df["Delta_Sharpe"] = (all_results_df["Sharpe_dd"] - all_results_df["Sharpe_base"]).round(3)
all_results_df["Delta_MDD"] = (all_results_df["MDD_dd"] - all_results_df["MDD_base"]).round(3)
all_results_df["Delta_CAGR"] = (all_results_df["CAGR_dd"] - all_results_df["CAGR_base"]).round(3)
all_results_df["Delta_Calmar"] = (all_results_df["Calmar_dd"] - all_results_df["Calmar_base"]).round(3)

all_results_df

In [ ]:
all_results_df.sort_values(by="Sharpe_dd", ascending=False)

In [ ]:
all_results_df.sort_values(by="Calmar_dd", ascending=False)

In [ ]:
all_results_df.sort_values(by="Delta_MDD", ascending=False)

In [ ]:
focus_df = all_results_df[
    all_results_df["Strategy"].str.contains("ls_20|blend", case=False, regex=True)
].copy()

focus_df.sort_values(by="Sharpe_dd", ascending=False)

**Only LS20 Family & Blends**

In [ ]:
focus_df.sort_values(by="Calmar_dd", ascending=False)

In [ ]:
out_dir = PROJECT_ROOT / "experiments/completed/exp_005_risk_engine_v1"
out_dir.mkdir(parents=True, exist_ok=True)

all_results_df.to_csv(out_dir / "dd_overlay_all_strategies_comparison.csv", index=False)
focus_df.to_csv(out_dir / "dd_overlay_focus_ls20_blends.csv", index=False)

**Tuning On Multiple Exposures**

In [ ]:
def drawdown_exposure_custom(drawdown, levels):
    e1, e2, e3, e4 = levels

    exposure = pd.Series(drawdown.index, dtype=float)

    exposure[drawdown > -0.10] = e1
    exposure[(drawdown <= -0.10) & (drawdown > -0.20)] = e2
    exposure[(drawdown <= 0.20) & (drawdown > -0.30)] = e3
    exposure[(drawdown <= 0.30)] = e4

    return exposure

In [ ]:
exposure_sets = {
    "aggressive": [1.0, 0.75, 0.50, 0.25],
    "mild": [1.0, 0.85, 0.70, 0.55],
    "very_mild": [1.0, 0.90, 0.80, 0.70],
    "floor_60": [1.0, 0.90, 0.80, 0.60]

}

In [ ]:
tuning_resuts =[]

for strat_name in [
    "ls_20pct_plus_sqrt_partial_normalized",
    "ls_20pct_plus_sqrt_partial",
    "ls_20pct",
    "blend_60_40_ls20_plus_ls30",
    "ls_30pct"
]:
    df = all_strategies[strat_name]
    returns = df["portfolio_return"].fillna(0)
    equity = (1 + returns).cumprod()

    for label, levels in exposure_sets.items():
        dd = compute_drawdown(equity)
        exp = drawdown_exposure_custom(dd, levels)
        ret_dd = apply_exposure_to_return(returns, exp)
        equity_dd = (1 + ret_dd).cumprod()

        tuning_resuts.append({
            "strategy": strat_name,
            "Rule": label,
            "Sharpe": sharpe_ratio(ret_dd),
            "MDD": max_drawdown(equity_dd),
            "CAGR": compute_cagr(equity_dd),
            "Calmar": calmar_ratio(equity_dd),
            "Avg_Exposure": exp.shift(1).fillna(1.0).mean()
        })

tuning_df = pd.DataFrame(tuning_resuts).round(3)
tuning_df.sort_values(by="Sharpe", ascending=False)

“We evaluated multiple drawdown-based exposure rules (aggressive, mild, floor-based, and very mild) across our top-performing strategies from Project 04. The results show a clear trade-off between risk reduction and return preservation: aggressive drawdown control significantly reduced maximum drawdown (to ~-0.22) but severely impaired returns (CAGR ~0.09) due to very low average exposure (~25%). As the exposure rules were relaxed, both CAGR and Sharpe recovered, with the ‘very mild’ configuration achieving the best balance—maintaining high Sharpe ratios (~1.54), strong CAGR (0.29), and improved drawdown (-0.52) at a reasonable average exposure (~70%). Importantly, the LS 20% strategy family, particularly the sqrt partial normalized variant, consistently outperformed others under all configurations, demonstrating robustness to risk constraints. Overall, these findings highlight that proportional, less aggressive risk control preserves alpha more effectively, and that the optimal approach lies in balancing drawdown reduction with sustained market participation.”


**Adding Volatility Targeting**

In [ ]:
best_name = "ls_20pct_plus_sqrt_partial_normalized"

df_best = all_strategies[best_name].copy()

returns = df_best["portfolio_return"].fillna(0)
equity = (1 + returns).cumprod()

In [ ]:
dd = compute_drawdown(equity)
dd_exposure = drawdown_exposure_custom(dd, [1.0, 0.90, 0.80, 0.70])

dd_exposure.shift(1).fillna(1.0).describe()

In [ ]:
vol_target = 0.20

rolling_vol = returns.rolling(20).std() * np.sqrt(252)
vol_scale = (vol_target / rolling_vol).clip(0.5, 1.5)
vol_scale = vol_scale.fillna(1.0)

In [ ]:
final_exposure = dd_exposure * vol_scale
final_exposure = final_exposure.clip(0.60, 1.00)

In [ ]:
ret_dd_vol = apply_exposure_to_return(returns, final_exposure)
equity_dd_vol = (1 + ret_dd_vol).cumprod()

**Comparison baseline with DD - only vs DD + Vol**

In [ ]:
ret_dd_only = apply_exposure_to_return(returns, dd_exposure)
equity_dd_only = (1 + ret_dd_only).cumprod()

comparison_best = pd.DataFrame([
    {
        "Strategy": f"{best_name}_baseline",
        "Sharpe": sharpe_ratio(returns),
        "MDD": max_drawdown(equity),
        "CAGR": compute_cagr(equity),
        "Vol": annual_vol(returns),
        "Calmar": calmar_ratio(equity),
        "Avg_Exposure": 1.0,
    },
    {
        "Strategy": f"{best_name}_dd_only",
        "Sharpe": sharpe_ratio(ret_dd_only),
        "MDD": max_drawdown(equity_dd_only),
        "CAGR": compute_cagr(equity_dd_only),
        "Vol": annual_vol(ret_dd_only),
        "Calmar": calmar_ratio(equity_dd_only),
        "Avg_Exposure": dd_exposure.shift(1).fillna(1.0).mean(),
    },
    {
        "Strategy": f"{best_name}_dd_plus_vol",
        "Sharpe": sharpe_ratio(ret_dd_vol),
        "MDD": max_drawdown(equity_dd_vol),
        "CAGR": compute_cagr(equity_dd_vol),
        "Vol": annual_vol(ret_dd_vol),
        "Calmar": calmar_ratio(equity_dd_vol),
        "Avg_Exposure": final_exposure.shift(1).fillna(1.0).mean(),
    },
]).round(3)

comparison_best

**Plot All 3 Equity Curves**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(equity, label="Baseline")
plt.plot(equity_dd_only, label="DD only (very_mild)")
plt.plot(equity_dd_vol, label="DD + Vol Target")
plt.legend()
plt.title("LS20 + Sqrt Partial Normalized: Baseline vs DD vs DD+Vol")
plt.xlabel("Time")
plt.ylabel("Equity Curve")
plt.show()

Combining volatility targeting with the tuned drawdown overlay did not materially improve the strategy beyond drawdown control alone. Although the additional volatility layer slightly reduced maximum drawdown, it also lowered Sharpe, CAGR, and Calmar, while the equity curve remained largely a scaled-down version of the baseline path. This suggests that, in its current form, volatility targeting is acting mainly as an additional exposure reduction rather than providing a distinct risk-management benefit. As a result, the tuned very mild drawdown overlay remains the stronger and more efficient Project 05 configuration so far

**Conditional Risk Control**

In [ ]:
dd = compute_drawdown(equity)

# base DD exposure (very mild)
dd_exposure = drawdown_exposure_custom(dd, [1.0, 0.90, 0.80, 0.70])

# volatility scaling
rolling_vol = returns.rolling(20).std() * np.sqrt(252)
vol_scale = (0.20 / rolling_vol).clip(0.5, 1.5)
vol_scale = vol_scale.fillna(1.0)

# CONDITION: only apply vol targeting in deep drawdown
threshold = -0.20

final_exposure_cond = dd_exposure.copy()

mask = dd <= threshold

final_exposure_cond[mask] = (
    dd_exposure[mask] * vol_scale[mask]
).clip(0.60, 1.00)

In [ ]:
ret_dd_vol_cond = apply_exposure_to_return(returns, final_exposure_cond)
equity_dd_vol_cond = (1 + ret_dd_vol_cond).cumprod() 

In [ ]:
ret_dd_only = apply_exposure_to_return(returns, dd_exposure)
equity_dd_only = (1 + ret_dd_only).cumprod()

comparison_best = pd.DataFrame([
    {
        "Strategy": f"{best_name}_baseline",
        "Sharpe": sharpe_ratio(returns),
        "MDD": max_drawdown(equity),
        "CAGR": compute_cagr(equity),
        "Vol": annual_vol(returns),
        "Calmar": calmar_ratio(equity),
        "Avg_Exposure": 1.0,
    },
    {
        "Strategy": f"{best_name}_dd_only",
        "Sharpe": sharpe_ratio(ret_dd_only),
        "MDD": max_drawdown(equity_dd_only),
        "CAGR": compute_cagr(equity_dd_only),
        "Vol": annual_vol(ret_dd_only),
        "Calmar": calmar_ratio(equity_dd_only),
        "Avg_Exposure": dd_exposure.shift(1).fillna(1.0).mean(),
    },
    {
        "Strategy": f"{best_name}_dd_plus_vol",
        "Sharpe": sharpe_ratio(ret_dd_vol),
        "MDD": max_drawdown(equity_dd_vol),
        "CAGR": compute_cagr(equity_dd_vol),
        "Vol": annual_vol(ret_dd_vol),
        "Calmar": calmar_ratio(equity_dd_vol),
        "Avg_Exposure": final_exposure.shift(1).fillna(1.0).mean(),
    },
    {
    "Strategy": f"{best_name}_dd_plus_vol_conditional", # applying vol targeting only where dds are high
        "Sharpe": sharpe_ratio(ret_dd_vol_cond),
        "MDD": max_drawdown(equity_dd_vol_cond),
        "CAGR": compute_cagr(equity_dd_vol_cond),
        "Vol": annual_vol(ret_dd_vol_cond),
        "Calmar": calmar_ratio(equity_dd_vol_cond),
        "Avg_Exposure": final_exposure_cond.shift(1).fillna(1.0).mean(),
    },
]).round(3)

comparison_best

**Smooth DD Control**

In [ ]:
def drawdown_exposure_smooth(drawdown, floor=0.60, k=3.0):
    """
    Smooth exposure: higher drawdown -> gradually lower exposure.
    floor: minimum exposure
    k: curvature (higher = more aggressive)
    """

# drawdown is negative
    dd_mag = drawdown.abs()

#smooth decay: exp(-k * dd)
    exp_scale = np.exp(-k * dd_mag)

# map to [floor, 1.0]
    exposure = floor +(1 - floor) * exp_scale

    return exposure


In [ ]:
dd = compute_drawdown(equity)
exp_smooth = drawdown_exposure_smooth(dd, floor=0.60, k=3.0)

ret_smooth = apply_exposure_to_return(returns, exp_smooth)
eq_smooth = (1 + ret_smooth).cumprod()

**Recovery**

In [ ]:
def apply_recovery_boost(exposure, drawdown, recovery_speed=0.15):
    """
    If drawdown is improving, increase exposure slightly
    """

    dd_change = drawdown.diff() #positive = recovering

    boost = (dd_change > 0) * recovery_speed
    boosted_exposure = exposure + boost

    return boosted_exposure.clip(0.60, 1.0)

In [ ]:
exp_smooth_recovery = apply_recovery_boost(exp_smooth, dd, recovery_speed=0.15)

ret_smooth_recovery = apply_exposure_to_return(returns, exp_smooth_recovery)
eq_smooth_recovery = (1 + ret_smooth_recovery).cumprod()

**Regime Filter**

In [ ]:
df_regime = pd.read_csv(
    PROJECT_ROOT / "data/processed/vol_regime_calibrated_oof.csv"
)

In [ ]:
# assuming you already have this from Project 02
# p_high_vol = probability of high volatility regime

def regime_exposure(p_high_vol, alpha=0.7):
    return (1 - p_high_vol) ** alpha

In [ ]:
df_best = df_best.reset_index(drop=True).copy()
df_regime = df_regime.reset_index(drop=True).copy()

print("df_best:", len(df_best))
print("df_regime:", len(df_regime))

df_regime_sub = df_regime.tail(len(df_best)).reset_index(drop=True)

print("df_regime_sub:", len(df_regime_sub))

df_best["p_high_vol_calibrated"] = df_regime_sub["p_high_vol_calibrated"].values

In [ ]:
p_high_vol = df_best["p_high_vol_calibrated"]

exp_regime = regime_exposure(p_high_vol).shift(1).fillna(1.0)

exp_combined = exp_smooth * (0.5 + 0.5 * exp_regime)
exp_combined = exp_combined.clip(0.60, 1.0)

ret_combined = apply_exposure_to_return(returns, exp_combined)
eq_combined = (1 + ret_combined).cumprod()

In [ ]:
df_best[["p_high_vol_calibrated"]].isna().sum()

**Multi Strategy Allocation**

In [ ]:
returns_1 = all_strategies["ls_20pct"]["portfolio_return"]
returns_2 = all_strategies["ls_20pct_plus_sqrt_partial_normalized"]["portfolio_return"]
returns_3 = all_strategies["blend_60_40_ls20_plus_ls30"]["portfolio_return"]

combined_returns = (
    0.4 * returns_1 +
    0.4 * returns_2 +
    0.2 * returns_3
).fillna(0)

eq_multi = (1 + combined_returns).cumprod()

In [ ]:
dd_multi = compute_drawdown(eq_multi)
exp_multi = drawdown_exposure_smooth(dd_multi, floor=0.60)

ret_multi_dd = apply_exposure_to_return(combined_returns, exp_multi)
eq_multi_dd = (1 + ret_multi_dd).cumprod()

**Comparison**

In [ ]:
comparison_best = pd.DataFrame([
    {
        "Strategy": f"{best_name}_baseline",
        "Sharpe": sharpe_ratio(returns),
        "MDD": max_drawdown(equity),
        "CAGR": compute_cagr(equity),
        "Vol": annual_vol(returns),
        "Calmar": calmar_ratio(equity),
        "Avg_Exposure": 1.0,
    },
    {
        "Strategy": f"{best_name}_dd_only",
        "Sharpe": sharpe_ratio(ret_dd_only),
        "MDD": max_drawdown(equity_dd_only),
        "CAGR": compute_cagr(equity_dd_only),
        "Vol": annual_vol(ret_dd_only),
        "Calmar": calmar_ratio(equity_dd_only),
        "Avg_Exposure": dd_exposure.shift(1).fillna(1.0).mean(),
    },
    {
        "Strategy": f"{best_name}_smooth_dd",
        "Sharpe": sharpe_ratio(ret_smooth),
        "MDD": max_drawdown(eq_smooth),
        "CAGR": compute_cagr(eq_smooth),
        "Vol": annual_vol(ret_smooth),
        "Calmar": calmar_ratio(eq_smooth),
        "Avg_Exposure": exp_smooth.shift(1).fillna(1.0).mean(),
    },
    {
        "Strategy": f"{best_name}_smooth_dd_plus_regime",
        "Sharpe": sharpe_ratio(ret_combined),
        "MDD": max_drawdown(eq_combined),
        "CAGR": compute_cagr(eq_combined),
        "Vol": annual_vol(ret_combined),
        "Calmar": calmar_ratio(eq_combined),
        "Avg_Exposure": exp_combined.shift(1).fillna(1.0).mean(),
    },
]).round(3)

comparison_best

Applying a smooth, continuous drawdown-based exposure control significantly improved the risk-adjusted performance of the strategy by increasing Sharpe and Calmar while maintaining similar CAGR, whereas adding a volatility regime layer provided only marginal drawdown improvement at the cost of reduced returns and Sharpe, indicating limited incremental value.

**Tuning Smooth DD - ls_20pct_plus_sqrt_partial_normalized**

In [ ]:
k_values = [2, 3, 4, 5]
floor_values = [0.55, 0.60, 0.65, 0.70]

In [ ]:
tuning_results = []

for k in k_values:
    for floor in floor_values:
        
        # smooth DD exposure
        exp_smooth = drawdown_exposure_smooth(dd, floor=floor, k=k)
        
        # apply exposure
        ret_smooth = apply_exposure_to_return(returns, exp_smooth)
        eq_smooth = (1 + ret_smooth).cumprod()
        
        tuning_results.append({
            "k": k,
            "floor": floor,
            "Sharpe": sharpe_ratio(ret_smooth),
            "MDD": max_drawdown(eq_smooth),
            "CAGR": compute_cagr(eq_smooth),
            "Vol": annual_vol(ret_smooth),
            "Calmar": calmar_ratio(eq_smooth),
            "Avg_Exposure": exp_smooth.shift(1).fillna(1.0).mean(),
        })

tuning_df = pd.DataFrame(tuning_results).round(3)

In [ ]:
tuning_df.sort_values(by="Sharpe", ascending=False)

The tuning of the smooth drawdown-based exposure control revealed that increasing the aggressiveness of the response (higher k) and allowing a lower minimum exposure (lower floor) materially improved risk-adjusted performance. In particular, configurations around k=5 and floor=0.55 delivered the strongest results, achieving a significant increase in Sharpe ratio (up to ~1.88) and Calmar ratio (~0.85) while maintaining nearly unchanged CAGR (~42–43%) and reducing maximum drawdown to approximately -50% from the baseline ~-65%. Importantly, performance remained stable across nearby parameter combinations (k=4–5, floor=0.55–0.60), indicating that the improvement is not due to a fragile parameter choice but rather a robust region of the parameter space. Overall, the results demonstrate that a smooth, continuous drawdown control can meaningfully enhance risk-adjusted returns without sacrificing long-term compounding.

**Smooth DD on Multiple Strategies**

In [ ]:
BEST_K = 5
BEST_FLOOR = 0.55

strategy_base = PROJECT_ROOT / "experiments/completed/exp_004_project04_final/strategy_timeseries"

strategy_files = sorted(strategy_base.glob("*.csv"))

multi_results = []
controlled_returns = {}

for file in strategy_files:
    name = file.stem
    df = pd.read_csv(file)

    returns_i = df["portfolio_return"].fillna(0)
    equity_i = (1 + returns_i).cumprod()

    dd_i = compute_drawdown(equity_i)
    exp_i = drawdown_exposure_smooth(dd_i, floor=BEST_FLOOR, k=BEST_K)

    ret_i_dd = apply_exposure_to_return(returns_i, exp_i)
    eq_i_dd = (1 + ret_i_dd).cumprod()

    controlled_returns[name] = ret_i_dd

    multi_results.append({
        "Strategy": name,
        "Sharpe_base": sharpe_ratio(returns_i),
        "MDD_base": max_drawdown(equity_i),
        "CAGR_base": compute_cagr(equity_i),
        "Calmar_base": calmar_ratio(equity_i),
        
        "Sharpe_smooth_dd": sharpe_ratio(ret_i_dd),
        "MDD_smooth_dd": max_drawdown(eq_i_dd),
        "CAGR_smooth_dd": compute_cagr(eq_i_dd),
        "Calmar_smooth_dd": calmar_ratio(eq_i_dd),
        "Avg_Exposure": exp_i.shift(1).fillna(1.0).mean(),
    })

multi_results_df = pd.DataFrame(multi_results).round(3)

multi_results_df.sort_values(by="Sharpe_smooth_dd", ascending=False)

In [ ]:
multi_results_df.sort_values(by="Calmar_smooth_dd", ascending=False)

Applying the optimized smooth drawdown control across multiple strategies demonstrated consistent improvements in risk-adjusted performance, and combining selectively diversified strategies is expected to further enhance stability by reducing correlation-driven drawdowns without materially sacrificing long-term returns.

**Blending strategies - Equal weights**

One thing to remember here is we are not blending independent strategies but highly correlated variations of the same alpha

In [ ]:

import json
import sqlite3

db_path = PROJECT_ROOT / "storage/quant_lab.db"
conn = sqlite3.connect(db_path)

In [ ]:
config = json.loads(
    pd.read_sql("""
    SELECT config_json
    FROM configs
    WHERE experiment_id = 'exp_005_risk_engine_final'
    """, conn).loc[0, "config_json"]
)

weights = config["weights"]
selected = list(weights.keys())

In [ ]:
multi_ret = pd.concat(
    [controlled_returns[s] for s in selected],
    axis=1
).mean(axis=1)

multi_eq = (1 + multi_ret).cumprod()

In [ ]:
multi_summary = pd.DataFrame([{
    "Strategy": "Multi-strategy + smooth DD",
    "Sharpe": sharpe_ratio(multi_ret),
    "MDD": max_drawdown(multi_eq),
    "CAGR": compute_cagr(multi_eq),
    "Vol": annual_vol(multi_ret),
    "Calmar": calmar_ratio(multi_eq),
}]).round(3)

multi_summary

**Blending Weighted Portfolio**

In [ ]:
config = json.loads(
    pd.read_sql("""
    SELECT config_json
    FROM configs
    WHERE experiment_id = 'exp_005_risk_engine_final'
    """, conn).loc[0, "config_json"]
)

weights = config["weights"]
selected = list(weights.keys())

In [ ]:
# build weighted portfolio
df_multi = pd.concat(
    [controlled_returns[s] for s in selected],
    axis=1
)

df_multi.columns = selected  # name columns

# apply weights
multi_ret = sum(df_multi[col] * weights[col] for col in selected)

multi_eq = (1 + multi_ret).cumprod()

In [ ]:
multi_summary = pd.DataFrame([{
    "Strategy": "Multi-strategy + smooth DD",
    "Sharpe": sharpe_ratio(multi_ret),
    "MDD": max_drawdown(multi_eq),
    "CAGR": compute_cagr(multi_eq),
    "Vol": annual_vol(multi_ret),
    "Calmar": calmar_ratio(multi_eq),
}]).round(3)

multi_summary

Introducing a weighted multi-strategy allocation preserved the strength of the primary alpha while benefiting from diversification, resulting in improved CAGR relative to equal-weighting and reduced drawdown compared to the single-strategy approach, achieving a well-balanced and robust portfolio.

**Portfolio level DD control**

In [ ]:
# Portfolio-level drawdown
portfolio_dd = compute_drawdown(multi_eq)

# Apply same tuned smooth DD rule at portfolio level
portfolio_exp = drawdown_exposure_smooth(
    portfolio_dd,
    floor=0.55,
    k=5
)

# Apply exposure to combined portfolio returns
multi_ret_portfolio_dd = apply_exposure_to_return(multi_ret, portfolio_exp)
multi_eq_portfolio_dd = (1 + multi_ret_portfolio_dd).cumprod()

In [ ]:
portfolio_dd_summary = pd.DataFrame([
    {
        "Strategy": "Weighted multi-strategy + smooth DD",
        "Sharpe": sharpe_ratio(multi_ret),
        "MDD": max_drawdown(multi_eq),
        "CAGR": compute_cagr(multi_eq),
        "Vol": annual_vol(multi_ret),
        "Calmar": calmar_ratio(multi_eq),
        "Avg_Exposure": 1.0,
    },
    {
        "Strategy": "Weighted multi-strategy + strategy DD + portfolio DD",
        "Sharpe": sharpe_ratio(multi_ret_portfolio_dd),
        "MDD": max_drawdown(multi_eq_portfolio_dd),
        "CAGR": compute_cagr(multi_eq_portfolio_dd),
        "Vol": annual_vol(multi_ret_portfolio_dd),
        "Calmar": calmar_ratio(multi_eq_portfolio_dd),
        "Avg_Exposure": portfolio_exp.shift(1).fillna(1.0).mean(),
    },
]).round(3)

portfolio_dd_summary

In [ ]:
plt.plot(multi_eq, label="Before portfolio DD")
plt.plot(multi_eq_portfolio_dd, label="After portfolio DD")
plt.legend()

The portfolio-level drawdown control successfully reduces downside volatility and drawdowns while preserving the overall structure and recovery dynamics of the equity curve, indicating effective risk control without materially degrading the underlying signal.

**STACK**

Signal (Project 4)

           |

Strategy construction

           |

Strategy-level DD control

           |

Portfolio weighting

           |

Portfolio-level DD control

In [ ]:
out_dir = PROJECT_ROOT / "experiments/completed/exp_005_risk_engine_final"
out_dir.mkdir(parents=True, exist_ok=True)

portfolio_dd_summary.to_csv(out_dir / "final_project05_summary.csv", index=False)

pd.DataFrame({
    "portfolio_return": multi_ret_portfolio_dd,
    "equity_curve": multi_eq_portfolio_dd,
    "portfolio_dd_exposure": portfolio_exp.shift(1).fillna(1.0),
}).to_csv(out_dir / "final_weighted_multi_strategy_portfolio_dd.csv", index=False)

multi_results_df.to_csv(out_dir / "all_strategies_smooth_dd_results.csv", index=False)

In [ ]:
import json

config = {
    "project": "Project 05 Risk Engine",
    "final_model": "weighted_multi_strategy_with_strategy_and_portfolio_smooth_dd",
    "strategy_level_dd": {
        "method": "smooth_drawdown_exposure",
        "k": 5,
        "floor": 0.55
    },
    "portfolio_level_dd": {
        "method": "smooth_drawdown_exposure",
        "k": 5,
        "floor": 0.55
    },
    "weights": {
        "ls_20pct_plus_sqrt_partial_normalized": 0.5,
        "blend_60_40_ls20_plus_ls30": 0.3,
        "ls_30pct": 0.2
    }
}

with open(out_dir / "config.json", "w") as f:
    json.dump(config, f, indent=4)

In [ ]:
import sqlite3

db_path = PROJECT_ROOT / "storage/quant_lab.db"
conn = sqlite3.connect(db_path)

print("DB created at:", db_path)

**Create Tables - SQL**

In [ ]:
cursor = conn.cursor()

cursor.execute("""
               CREATE TABLE IF NOT EXISTS experiments (
               experiment_id TEXT PRIMARY KEY,
               project TEXT,
               desciption TEXT,
               created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
               )
               """)

cursor.execute("""
               CREATE TABLE IF NOT EXISTS stratrgy_metrics (
               experiment_id TEXT,
               strategy TEXT,
               sharpe REAL,
               cagr REAL,
               vol REAL,
               calmar REAL,
               avg_exposure REAL,
               PRIMARY KEY (experiment_id, strategy)
               )
               """)

cursor.execute("""
               CREATE TABLE IF NOT EXISTS portfolio_timeseries (
               experiment_id TEXT,
               date TEXT,
               portfolio_return REAL,
               equity_curve REAL,
               exposure REAL)
               """)

cursor.execute("""
               CREATE TABLE IF NOT EXISTS configs (
               experiment_id TEXT PRIMARY KEY,
               config_json TEXT
               )
               """)

conn.commit()

**Insert Project 05 Experiment Metadata**

In [ ]:
exp_id = "exp_005_risk_engine_final"

cursor.execute("""
               INSERT OR REPLACE INTO experiments (experiment_id, project, desciption)
               VALUES (?, ?, ?)
               """, (
                   exp_id,
                   "Project 05 Risk Engine",
                   "Weighted multi-strategy portfolio with strategy-level and portfolio-level smooth drawdown control"

               ))

conn.commit()

**Insert final Metrics**

In [ ]:
metrics_path = PROJECT_ROOT / "experiments/completed/exp_005_risk_engine_final/final_project05_summary.csv"
df_metrics = pd.read_csv(metrics_path)

df_metrics = df_metrics.rename(columns={
    "Strategy": "strategy",
    "Sharpe": "sharpe",
    "MDD": "mdd",
    "CAGR": "cagr",
    "Vol": "vol",
    "Calmar": "calmar",
    "Avg_Exposure": "avg_exposure"
})
df_metrics["experiment_id"] = exp_id

df_metrics = df_metrics[
    ["experiment_id", "strategy", "sharpe", "mdd", "cagr", "vol", "calmar", "avg_exposure"]
]

cursor.execute(
    "DELETE FROM strategy_metrics WHERE experiment_id = ?",
    (exp_id,)
)
conn.commit()

df_metrics.to_sql("strategy_metrics", conn, if_exists="append", index=False)

In [ ]:
print(df_metrics.columns.tolist())
df_metrics.head() # this a dataframe - from memory

In [ ]:
pd.read_sql("SELECT * FROM strategy_metrics", conn) #this read from database

**Insert Final Portfolio Time Series**

In [ ]:
ts_path = PROJECT_ROOT / "experiments/completed/exp_005_risk_engine_final/final_weighted_multi_strategy_portfolio_dd.csv"

df_ts = pd.read_csv(ts_path)

df_ts["experiment_id"] = exp_id

if "Date" not in df_ts.columns and "date" not in df_ts.columns:
    df_ts["date"] = df_ts.index.astype(str)

df_ts = df_ts.rename(columns={
    "Date": "date",
    "portfolio_dd_exposure": "exposure"
})

df_ts = df_ts[
    ["experiment_id", "date", "portfolio_return", "equity_curve", "exposure"]
]

cursor.execute(
    "DELETE FROM portfolio_timeseries WHERE experiment_id = ?",
    (exp_id,)
)
conn.commit()

df_ts.to_sql("portfolio_timeseries", conn, if_exists="append", index=False)

In [ ]:
config_path = PROJECT_ROOT / "experiments/completed/exp_005_risk_engine_final/config.json"

with open(config_path, "r") as f:
    config_json = json.dumps(json.load(f))

cursor.execute("""
               INSERT OR REPLACE INTO configs (experiment_id, config_json)
               VALUES (?, ?)
               """, (exp_id, config_json))


conn.commit()

**Verifying Inserts**

In [ ]:
pd.read_sql("SELECT * FROM experiments", conn)

In [ ]:
pd.read_sql("SELECT * FROM strategy_metrics", conn)

In [ ]:
pd.read_sql("""
            SELECT *
            FROM portfolio_timeseries
            WHERE experiment_id = 'exp_005_risk_engine_final'
            LIMIT 5
            """, conn)

In [ ]:
# Best strategy variant in Project 05
pd.read_sql("""
SELECT strategy, sharpe, mdd, cagr, calmar
FROM strategy_metrics
WHERE experiment_id = 'exp_005_risk_engine_final'
ORDER BY calmar DESC
""", conn)

In [ ]:
# Final equity value from stored time series
pd.read_sql("""
SELECT 
    experiment_id,
    MAX(CAST(date AS INTEGER)) AS last_step,
    equity_curve AS final_equity
FROM portfolio_timeseries
WHERE experiment_id = 'exp_005_risk_engine_final'
ORDER BY CAST(date AS INTEGER) DESC
LIMIT 1
""", conn)

In [ ]:
# Check stored config
pd.read_sql("""
SELECT config_json
FROM configs
WHERE experiment_id = 'exp_005_risk_engine_final'
""", conn)

In [ ]:
top_strategy_df = pd.read_sql("""
SELECT 
    strategy,
    sharpe,
    mdd,
    cagr,
    calmar
FROM strategy_metrics
WHERE experiment_id = 'exp_005_risk_engine_final'
ORDER BY calmar DESC
LIMIT 1
""", conn)

top_strategy_df

In [ ]:
best_strategy_from_db = top_strategy_df.loc[0, "strategy"]
best_strategy_from_db

In [ ]:
def get_top_strategies(conn, experiment_id, metric="calmar", limit=3):
    allowed_metrics = {"sharpe", "mdd", "cagr", "vol", "calmar", "avg_exposure"}
    
    if metric not in allowed_metrics:
        raise ValueError(f"metric must be one of {allowed_metrics}")
    
    query = f"""
    SELECT 
        strategy,
        sharpe,
        mdd,
        cagr,
        vol,
        calmar,
        avg_exposure
    FROM strategy_metrics
    WHERE experiment_id = ?
    ORDER BY {metric} DESC
    LIMIT ?
    """
    
    return pd.read_sql(query, conn, params=(experiment_id, limit))

In [ ]:
# By Calmar
get_top_strategies(
    conn,
    experiment_id="exp_005_risk_engine_final",
    metric="calmar",
    limit=2
)

In [ ]:
# By Sharpe
get_top_strategies(
    conn,
    experiment_id="exp_005_risk_engine_final",
    metric="sharpe",
    limit=2
)


In [ ]:
selected_strategies = get_top_strategies(
    conn,
    experiment_id="exp_005_risk_engine_final",
    metric="calmar",
    limit=3
)["strategy"].tolist()

selected_strategies

In [ ]:
import json

config_df = pd.read_sql("""
SELECT config_json
FROM configs
WHERE experiment_id = 'exp_005_risk_engine_final'
""", conn)

config_json_str = config_df.loc[0, "config_json"]
config = json.loads(config_json_str)

config

**Load config from DB**

In [ ]:
import json

config_df = pd.read_sql("""
SELECT config_json
FROM configs
WHERE experiment_id = 'exp_005_risk_engine_final'
""", conn)

config = json.loads(config_df.loc[0, "config_json"])

In [ ]:
weights = config["weights"]
selected = list(weights.keys())

In [ ]:
print(selected)
print(weights)

**Auto Select The Best Experiment**

In [ ]:
# 1) Load base portfolio weights
base_exp_id = "exp_005_risk_engine_final"

base_config_df = pd.read_sql("""
SELECT config_json
FROM configs
WHERE experiment_id = ?
""", conn, params=(base_exp_id,))

base_config = json.loads(base_config_df.loc[0, "config_json"])

weights = base_config["weights"]
selected = list(weights.keys())

In [ ]:
# 2) Build base weighted portfolio
df_multi = pd.concat(
    [controlled_returns[s] for s in selected],
    axis=1
)

df_multi.columns = selected

multi_ret = sum(df_multi[col] * weights[col] for col in selected)
multi_eq = (1 + multi_ret).cumprod()

In [ ]:
import re

def extract_k_floor(exp_id):
    match = re.search(r"_k(\d+)_floor([0-9_]+)", exp_id)
    if not match:
        return None, None

    k = float(match.group(1))
    floor = float(match.group(2).replace("_", "."))
    return k, floor

In [ ]:
# 3) Get best DD parameter experiment
best_param = pd.read_sql("""
SELECT experiment_id, calmar
FROM strategy_metrics
WHERE experiment_id LIKE 'auto_smooth_dd_%'
ORDER BY calmar DESC
LIMIT 1
""", conn)

best_exp_id = best_param.loc[0, "experiment_id"]

k, floor = extract_k_floor(best_exp_id)
k, floor

In [ ]:
# 4) Apply best DD overlay
portfolio_dd = compute_drawdown(multi_eq)

portfolio_exp = drawdown_exposure_smooth(
    portfolio_dd,
    floor=floor,
    k=k
)

multi_ret_final = apply_exposure_to_return(multi_ret, portfolio_exp)
multi_eq_final = (1 + multi_ret_final).cumprod()

In [ ]:
# 5) Evaluate final result
final_summary = pd.DataFrame([{
    "Strategy": "Final Portfolio: DB weights + best DD params",
    "k": k,
    "floor": floor,
    "Sharpe": sharpe_ratio(multi_ret_final),
    "MDD": max_drawdown(multi_eq_final),
    "CAGR": compute_cagr(multi_eq_final),
    "Vol": annual_vol(multi_ret_final),
    "Calmar": calmar_ratio(multi_eq_final),
    "Avg_Exposure": portfolio_exp.shift(1).fillna(1.0).mean()
}]).round(3)

final_summary

In [ ]:
# 4) Apply best DD overlay
portfolio_dd = compute_drawdown(multi_eq)

portfolio_exp = drawdown_exposure_smooth(
    portfolio_dd,
    floor=floor,
    k=k
)

multi_ret_final = apply_exposure_to_return(multi_ret, portfolio_exp)
multi_eq_final = (1 + multi_ret_final).cumprod()

**Apply Portfolio-level DD to the DB reconstructed Portfolio**

In [ ]:
portfolio_dd = compute_drawdown(multi_eq)


portfolio_exp = drawdown_exposure_smooth(
    portfolio_dd,
    floor=0.55,
    k=5
)

multi_ret_portfolio_dd = apply_exposure_to_return(multi_ret, portfolio_exp)
multi_eq_portfolio_dd = (1 + multi_ret_portfolio_dd).cumprod()

In [ ]:
# 5) Evaluate final result
final_summary = pd.DataFrame([{
    "Strategy": "Final Portfolio: DB weights + best DD params",
    "k": k,
    "floor": floor,
    "Sharpe": sharpe_ratio(multi_ret_final),
    "MDD": max_drawdown(multi_eq_final),
    "CAGR": compute_cagr(multi_eq_final),
    "Vol": annual_vol(multi_ret_final),
    "Calmar": calmar_ratio(multi_eq_final),
    "Avg_Exposure": portfolio_exp.shift(1).fillna(1.0).mean()
}]).round(3)

final_summary

In [ ]:
multi_exp_portfolio_dd_summary = pd.DataFrame([{
    "Strategy": "DB-selected multi-experiment portfolio + portfolio DD",
    "Sharpe": sharpe_ratio(multi_ret_portfolio_dd),
    "MDD": max_drawdown(multi_eq_portfolio_dd),
    "CAGR": compute_cagr(multi_eq_portfolio_dd),
    "Vol": annual_vol(multi_ret_portfolio_dd),
    "Calmar": calmar_ratio(multi_eq_portfolio_dd),
    "Avg_Exposure": portfolio_exp.shift(1).fillna(1.0).mean()
}]).round(3)

multi_exp_portfolio_dd_summary

**Store This as New Experiment**

In [ ]:
new_exp_id = "exp_006_multi_experiment_ensemble"

In [ ]:
cursor.execute("""
INSERT OR REPLACE INTO experiments (experiment_id, project)
VALUES (?, ?)
""", (
    new_exp_id,
    "Project 05 Risk Engine"
))

conn.commit()

In [ ]:

#store metrics

df_metrics = multi_exp_portfolio_dd_summary.copy()
df_metrics["experiment_id"] = new_exp_id

df_metrics = df_metrics.rename(columns={
    "Strategy": "strategy",
    "Sharpe": "sharpe",
    "MDD": "mdd",
    "CAGR": "cagr",
    "Vol": "vol",
    "Calmar": "calmar",
    "Avg_Exposure": "avg_exposure"
})

df_metrics = df_metrics[
    ["experiment_id", "strategy", "sharpe", "mdd", "cagr", "vol", "calmar", "avg_exposure"]
]

# delete old if exists
cursor.execute("DELETE FROM strategy_metrics WHERE experiment_id = ?", (new_exp_id,))
conn.commit()

df_metrics.to_sql("strategy_metrics", conn, if_exists="append", index=False)

In [ ]:
# store time series

df_ts = pd.DataFrame({
    "experiment_id": new_exp_id,
    "date": multi_eq_portfolio_dd.index.astype(str),
    "portfolio_return": multi_ret_portfolio_dd.values,
    "equity_curve": multi_eq_portfolio_dd.values,
    "exposure": portfolio_exp.shift(1).fillna(1.0).values
})

cursor.execute("DELETE FROM portfolio_timeseries WHERE experiment_id = ?", (new_exp_id,))
conn.commit()

df_ts.to_sql("portfolio_timeseries", conn, if_exists="append", index=False)

In [ ]:
# only use portfolio experiments
top_exps = pd.read_sql("""
SELECT 
    experiment_id,
    MAX(calmar) AS best_calmar
FROM strategy_metrics
WHERE experiment_id NOT LIKE 'auto_smooth_dd_%'
GROUP BY experiment_id
ORDER BY best_calmar DESC
LIMIT 3
""", conn)

top_experiment_ids = top_exps["experiment_id"].tolist()

In [ ]:
# # store config

# ensemble_config = {
#     "type": "multi_experiment_ensemble",
#     "source_experiments": top_experiment_ids,
#     "combined_weights": combined_weights,
#     "portfolio_level_dd": {
#         "method": "smooth_drawdown_exposure",
#         "k": 5,
#         "floor": 0.55
#     }
# }

# cursor.execute("""
# INSERT OR REPLACE INTO configs (experiment_id, config_json)
# VALUES (?, ?)
# """, (new_exp_id, json.dumps(ensemble_config)))

# conn.commit()

**Automated Experiment Generation**

automated experiment generation let the system create and test many research variants systematically instead of manually coding each one

In [ ]:
#define ranges

k_values = [3, 4, 5, 6]
floor_values = [0.50, 0.55, 0.60, 0.65]

In [ ]:
# build grid automatically

experiment_grid = [
    {"k": k, "floor": floor}
    for k in k_values
    for floor in floor_values
]

In [ ]:
# functions to run one experiment

def run_smooth_dd_experiment(exp_id, k, floor, returns):
    equity = (1 + returns).cumprod()

    dd = compute_drawdown(equity)
    exposure = drawdown_exposure_smooth(dd, floor=floor, k=k)

    ret_dd = apply_exposure_to_return(returns, exposure)
    eq_dd = (1 + ret_dd).cumprod()

    metrics = {
        "experiment_id": exp_id,
        "strategy": f"smooth_dd_k{k}_floor{floor}",
        "sharpe": sharpe_ratio(ret_dd),
        "mdd": max_drawdown(eq_dd),
        "vol": annual_vol(ret_dd),
        "calmar": calmar_ratio(eq_dd),
        "avg_exposure": exposure.shift(1).fillna(1.0).mean(),
    }

    ts = pd.DataFrame({
        "experiment_id": exp_id,
        "date": ret_dd.index.astype(str),
        "portfolio_return": ret_dd.values,
        "equity_curve": eq_dd.values,
        "exposure": exposure.shift(1).fillna(1.0).mean(),
    })

    config = {
        "type": "auto_generated_smooth_dd",
        "k": k,
        "floor": floor,
        "base_strategy": best_name,
    }

    return metrics, ts, config

In [ ]:
# Run and store experiments
for params in experiment_grid:
    k = params["k"]
    floor = params["floor"]

    exp_id = f"auto_smooth_dd_k{k}_floor{str(floor).replace('.', '_')}"

    metrics, ts, config = run_smooth_dd_experiment(
        exp_id=exp_id,
        k=k,
        floor=floor,
        returns=multi_ret
    )

    cursor.execute("""
                   INSERT OR REPLACE INTO experiments (experiment_id, project)
                   VALUES (?, ?)
                   """, (exp_id, "Project 05 Auto Experiments"))
    
    cursor.execute("DELETE FROM strategy_metrics WHERE experiment_id = ?", (exp_id,))
    cursor.execute("DELETE FROM portfolio_timeseries WHERE experiment_id = ?", (exp_id,))

    pd.DataFrame([metrics]).to_sql(
        "strategy_metrics",
        conn,
        if_exists="append",
        index=False
    )

    ts.to_sql(
        "portfolio_timeseries",
        conn,
        if_exists="append",
        index=False
    )

    cursor.execute("""
                   INSERT OR REPLACE INTO configs (experiment_id, config_json)
                   VALUES (?, ?)
                   """, (exp_id, json.dumps(config)))
    
    conn.commit()

In [ ]:
#order best generated experiment

pd.read_sql("""
            SELECT 
            experiment_id,
            strategy,
            sharpe,
            mdd,
            vol,
            calmar,
            avg_exposure
            FROM strategy_metrics
            WHERE experiment_id LIKE 'auto_smooth_dd_%'
            ORDER BY calmar DESC
            """, conn)

In [ ]:
df = pd.read_sql("""
SELECT 
    CAST(
        substr(
            experiment_id,
            instr(experiment_id, '_k') + 2,
            instr(experiment_id, '_floor') - (instr(experiment_id, '_k') + 2)
        ) AS INTEGER
    ) AS k,

    CAST(
        REPLACE(
            substr(experiment_id, instr(experiment_id, '_floor') + 6),
            '_',
            '.'
        ) AS FLOAT
    ) AS floor,

    calmar

FROM strategy_metrics
WHERE experiment_id LIKE 'auto_smooth_dd_%'
""", conn)

df

In [ ]:
df.columns

In [ ]:
#pivot Table

pivot = df.pivot_table(
    index="k",
    columns="floor",
    values="calmar",
    aggfunc="mean"   # or "max"
)

pivot

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(pivot, aspect='auto')
plt.colorbar()
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.title("Calmar Heatmap (k vs floor)")
plt.xlabel("floor")
plt.ylabel("k")
plt.show()

**Refactoring the DB schema**

In [ ]:
# create a table

cursor.execute("""
               CREATE TABLE IF NOT EXISTS risk_parameter_experiments (
               experiment_id TEXT PRIMARY KEY,
               project TEXT,
               k REAL,
               floor REAL,
               sharpe REAL,
               mdd REAL,
               cagr REAL,
               vol REAL,
               calmar REAL,
               avg_exposure
               )
               """)

conn.commit()

In [ ]:
# Insert autoexperiment results

df_auto = pd.read_sql("""
                      SELECT
                        experiment_id,
                        strategy,
                        sharpe,
                        mdd,
                        cagr,
                        vol,
                        calmar,
                        avg_exposure
                      FROM strategy_metrics
                      WHERE experiment_id LIKE 'auto_smooth_dd_%'
                      """, conn)

In [ ]:
# Extract k and floor

import re

def extract_k_floor(exp_id):
    match = re.search(r"_k(\d+)_floor([0-9_]+)", exp_id)
    if not match:
        return None, None
    
    k = float(match.group(1))
    floor = float(match.group(2).replace("_", "."))
    return k, floor

df_auto[["k", "floor"]] = df_auto["experiment_id"].apply(
    lambda x: pd.Series(extract_k_floor(x))
)

In [ ]:
#prepare clean table

df_risk_params = df_auto[[
    "experiment_id",
    "k",
    "floor",
    "sharpe",
    "mdd",
    "cagr",
    "vol",
    "calmar",
    "avg_exposure"
]].copy()

df_risk_params["project"] = "Project 05 Risk Engine"

df_risk_params = df_risk_params[[
    "experiment_id",
    "project",
    "k",
    "floor",
    "sharpe",
    "mdd",
    "cagr",
    "vol",
    "calmar",
    "avg_exposure"
]]

In [ ]:
cursor.execute("DROP TABLE IF EXISTS risk_parameter_experiments")
conn.commit()

In [ ]:
cursor.execute("""
CREATE TABLE risk_parameter_experiments (
    experiment_id TEXT PRIMARY KEY,
    project TEXT,
    k REAL,
    floor REAL,
    sharpe REAL,
    mdd REAL,
    cagr REAL,
    vol REAL,
    calmar REAL,
    avg_exposure REAL
)
""")
conn.commit()

In [ ]:
df_risk_params.to_sql(
    "risk_parameter_experiments",
    conn,
    if_exists="append",
    index=False
)

In [ ]:
pd.read_sql("PRAGMA table_info(risk_parameter_experiments);", conn)

In [ ]:
pd.read_sql("""
            SELECT k, floor, calmar
            FROM risk_parameter_experiments
            ORDER BY calmar DESC
            """, conn)

The drawdown-aware risk overlay was extended into a systematic parameter exploration framework, where combinations of smoothing intensity (k) and minimum exposure floor were evaluated and stored in a structured SQL database. Results show a clear pattern: higher smoothing (k ≈ 5–6) and lower exposure floors (≈ 0.50) consistently deliver superior Calmar ratios, with the best configuration (k=6, floor=0.50) achieving the strongest risk-adjusted performance. This indicates that aggressive drawdown control—particularly higher floors that cut exposure too early—tends to degrade returns, while smoother, less reactive exposure scaling preserves the underlying alpha more effectively. Overall, the experiment suggests that the base signal is robust, and that risk overlays should be applied conservatively; excessive intervention reduces performance rather than improving it.

**Apply best parameters to the final portfolio**

In [ ]:
best_param = pd.read_sql("""
SELECT k, floor
FROM risk_parameter_experiments
ORDER BY calmar DESC
LIMIT 1
""", conn)

k = best_param.loc[0, "k"]
floor = best_param.loc[0, "floor"]

In [ ]:
portfolio_dd = compute_drawdown(multi_eq)

portfolio_exp = drawdown_exposure_smooth(
    portfolio_dd,
    floor=floor,
    k=k
)

multi_ret_final = apply_exposure_to_return(multi_ret, portfolio_exp)
multi_eq_final = (1 + multi_ret_final).cumprod()

In [ ]:
final_comparison = pd.DataFrame([
    {
        "Strategy": "Base weighted portfolio",
        "Sharpe": sharpe_ratio(multi_ret),
        "MDD": max_drawdown(multi_eq),
        "CAGR": compute_cagr(multi_eq),
        "Vol": annual_vol(multi_ret),
        "Calmar": calmar_ratio(multi_eq),
    },
    {
        "Strategy": f"Optimized DD overlay k={k}, floor={floor}",
        "Sharpe": sharpe_ratio(multi_ret_final),
        "MDD": max_drawdown(multi_eq_final),
        "CAGR": compute_cagr(multi_eq_final),
        "Vol": annual_vol(multi_ret_final),
        "Calmar": calmar_ratio(multi_eq_final),
    }
]).round(3)

final_comparison

The drawdown-aware exposure overlay, when tuned using a systematic SQL-backed parameter search, significantly improved the portfolio’s risk-adjusted performance. The optimal configuration (k=6, floor=0.50) delivered a higher Sharpe ratio (2.15 vs 1.85), reduced maximum drawdown (−34% vs −48%), and improved the Calmar ratio (1.09 vs 0.81), while only marginally reducing CAGR. This indicates that a smooth, minimally restrictive exposure control preserves the underlying alpha while effectively mitigating downside risk. The consistency between the parameter search results and the final applied portfolio confirms the robustness of the research pipeline.